# M³Diff — Notebook FT: Fine-tune bằng dữ liệu tổng hợp

**Mục tiêu:** sửa 3 điểm yếu đã đo được — model không nhận ra **camera zoom/xoay/lia** (0/8 cặp zoom trên Real),
**đổi kích thước** (25,6% đúng loại), **dời vật ít** (26,7% ở mức 10%) — mà không làm tụt điểm OmniDiff.

| | |
|---|---|
| Dữ liệu train | ~2.700 cặp **tổng hợp** (nền OmniDiff *train*, vật COCO *không trùng* bộ test) + ~2.700 câu **OmniDiff train gốc** do người viết (để model không quên cách viết câu dài) |
| Train cái gì | LoRA r=64 trên LLM + toàn bộ MDP + projector, xuất phát từ checkpoint M³Diff. Vision tower giữ nguyên |
| Đánh giá (gốc vs fine-tune) | ① bộ test camera mới (240 cặp) ② bộ tổng hợp Notebook 1 (600 cặp) ③ OmniDiff-Real test ④ OmniDiff-Render test |
| Không lẫn train/test | Ảnh nền test ≠ train, vật thể test ≠ train, seed khác |

**Trước khi chạy:**
1. Compute profile: GPU **A100 80GB** (A100 40GB cũng được, tự giảm batch), CPU **8**, RAM **64 GiB**. Volume `m3diff` ở `/mnt/m3diff`.
2. Phải đã chạy xong **Notebook 1** (cần `/mnt/m3diff/synth/` và kết quả model gốc).
3. Chạy lần lượt từ trên xuống. **Không đổi cấu hình máy khi đang chạy.** Mọi kết quả ghi lên Volume, bị ngắt thì chạy lại Bước 1–5 rồi cell đang dở (train và eval đều chạy tiếp được).

**Thời gian / chi phí ước tính:** tải dữ liệu ~15 phút, sinh dữ liệu ~10 phút, train ~1–1,5 giờ, đánh giá ~1,5 giờ → tổng ~3–3,5 giờ.
Xem số $/hour ở Compute profile để nhân ra chi phí (ước khoảng $10–15).

## Bước 1 — Cấu hình và kiểm tra điều kiện

In [ ]:
import os, sys, json, shutil, subprocess, glob, time
VOL   = "/mnt/m3diff"
REPO  = "/root/M3Diff"
VENV  = "/root/m3diff-venv"
PY    = f"{VENV}/bin/python"
CKPT  = "/root/ckpt/M3Diff"
SYN   = f"{VOL}/synth"            # Notebook 1
FT    = f"{VOL}/ft"               # Notebook này
CODE  = f"{FT}/code"
OUT   = f"{FT}/m3diff-ft-lora"    # tên thư mục phải chứa 'm3diff' và 'lora' để repo nạp đúng kiểu LoRA
EVAL  = f"{FT}/eval"
OBJ_ALL   = f"{FT}/objects_all"
TRAIN_SYN = f"{FT}/train_synth"
CAMTEST   = f"{FT}/camtest"
MIX       = f"{FT}/train_mix.json"
QUANT, DTYPE = "none", "bfloat16"

# Siêu tham số
SCALE    = 1.0        # nhân số cặp train tổng hợp (1.0 ≈ 2.700 cặp)
N_REPLAY = 2700       # số câu OmniDiff train gốc trộn vào
LORA_R, LORA_ALPHA = 64, 128
LR, PROJ_LR, EPOCHS = 4e-5, 1e-5, 1
HF_TOKEN = os.environ.get("HF_TOKEN", "")
os.environ["MPLBACKEND"] = "Agg"
os.environ["TQDM_MININTERVAL"] = "20"
for d in [CODE, EVAL, f"{FT}/results"]:
    os.makedirs(d, exist_ok=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "KHÔNG CÓ GPU")
mem = int(gpu.split(",")[-1]) if gpu else 0
BATCH = 4 if mem >= 70000 else (2 if mem >= 39000 else 1)
ACCUM = 16 // BATCH
print(f"batch/GPU = {BATCH}, tích luỹ gradient = {ACCUM} (batch hiệu dụng 16)")
if mem < 39000:
    print("!! GPU dưới 40GB: train có thể hết bộ nhớ. Nên chọn A100.")

need = {"code Notebook 1": f"{SYN}/code/make_synth.py", "bộ test tổng hợp": f"{SYN}/data/manifest.jsonl",
        "câu model gốc trên bộ tổng hợp": f"{SYN}/data/answers.jsonl", "điểm model gốc (per_pair)": f"{SYN}/results/per_pair.csv",
        "vật thể test": f"{SYN}/objects/objects.csv", "kết quả gốc Real": f"{VOL}/results/OmniDiff_Real/OmniDiff_Real.jsonl"}
for k, v in need.items():
    print(("OK    " if os.path.exists(v) else "THIẾU ") + f"{k}: {v}")
print(("OK    " if os.path.exists(f"{VOL}/results/OmniDiff_Render/OmniDiff_Render.jsonl") else "(không có, bỏ qua Render) ")
      + "kết quả gốc Render")

## Bước 2 — Môi trường Python 3.10 (giống notebook eval) + thư viện train

In [ ]:
!{UV} pip install -q --python {PY} setuptools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"

In [ ]:
%uv pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}
if not os.path.exists(PY):
    !{UV} venv {VENV} --python 3.10 -q

%cd {REPO}
!{UV} pip install -q --python {PY} -e ".[eval]" "numpy<2" accelerate==0.29.1 bitsandbytes==0.43.3 install-jdk
!{PY} -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__); print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')"
!{UV} pip install -q --python {PY} peft==0.4.0 "numpy<2" "opencv-python-headless<4.12" pandas scipy matplotlib pycocotools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"

Kỳ vọng `peft 0.4.0`, `numpy 1.26.x`.

## Bước 3 — Tải checkpoint, ảnh OmniDiff (test + train) và file câu mô tả train gốc
Ảnh train dùng làm nền cho dữ liệu tổng hợp và cho phần câu OmniDiff gốc trộn vào. Lần đầu mất ~10–15 phút.

In [ ]:
print("HF_TOKEN:", "ĐÃ CÓ" if HF_TOKEN else "CHƯA CÓ — dễ bị chặn 429, xem hướng dẫn ở trên")
dl = f'''
import os, time
os.environ["HF_HUB_DISABLE_XET"] = "1"        # tải HTTP thường, tránh gọi xet-read-token cho từng file (hay bị 429)
from huggingface_hub import snapshot_download, hf_hub_download
tok = {HF_TOKEN!r} or None
jobs = [
    lambda: snapshot_download("IVC-liuyuan/M3Diff", local_dir="{CKPT}", token=tok, max_workers=4),
    lambda: snapshot_download("IVC-liuyuan/OmniDiff", repo_type="dataset", local_dir="{REPO}/datasets/OmniDiff",
                              allow_patterns=["Real/test/*", "Render/test/*", "Real/train/*", "Render/train/*"],
                              token=tok, max_workers=4),
    lambda: hf_hub_download("IVC-liuyuan/training_m3diff", "training_m3diff.json", repo_type="dataset",
                            local_dir="{FT}/orig", token=tok),
]
for j in jobs:
    for attempt in range(8):
        try:
            j(); break
        except Exception as e:
            if "429" not in str(e) or attempt == 7:
                raise
            print(f"  bị giới hạn 429, chờ 90 giây rồi tải tiếp (lần {{attempt+1}})", flush=True)
            time.sleep(90)
print("Tải xong")
'''
subprocess.run([PY, "-c", dl], check=True)
!du -sh {CKPT} {REPO}/datasets/OmniDiff/Real {REPO}/datasets/OmniDiff/Render {FT}/orig/training_m3diff.json
for s in ["Real", "Render"]:
    for sp in ["train", "test"]:
        print(s, sp, len(glob.glob(f"{REPO}/datasets/OmniDiff/{s}/{sp}/images/*")), "ảnh")

## Bước 4 — Patch script eval (giống notebook eval: nạp model lên GPU, chạy tiếp được)

In [ ]:
path = f"{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py"
src = open(path).read()

# --- Patch 1: nén model cho vừa T4 ---
old1 = """    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
    )"""
new1 = """    quant_mode = os.environ.get("M3DIFF_QUANT", "8bit")
    skip = ["vision_tower", "vision_resampler", "mm_projector", "lm_head"]
    extra = {}
    if quant_mode == "8bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_8bit=True, llm_int8_skip_modules=skip)
    elif quant_mode == "4bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=skip)
    print(f"[patch] quant_mode={quant_mode}", flush=True)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
        device_map={"": 0},
        **extra,
    )"""

# --- Patch 2: chạy tiếp từ chỗ dừng + ghi từng câu ---
old2 = '    ans_file = open(answers_file, "w")\n'
new2 = """    done_lines = []
    if os.path.exists(answers_file):
        for l in open(answers_file):
            try:
                json.loads(l)
                done_lines.append(l.strip())
            except Exception:
                pass
    done_ids = {json.loads(l)["id"] for l in done_lines}
    with open(answers_file, "w") as f:
        f.write("".join(l + "\\n" for l in done_lines))
    questions = [q for q in questions if q["id"] not in done_ids]
    print(f"[patch] resume: {len(done_ids)} done, {len(questions)} left", flush=True)
    ans_file = open(answers_file, "a")
"""
old3 = '        ans_file.flush()\n'
new3 = '        ans_file.close()\n        ans_file = open(answers_file, "a")\n'

if "M3DIFF_QUANT" not in src:
    assert old1 in src, "Không tìm thấy đoạn patch 1"
    src = src.replace(old1, new1)
if "[patch] resume" not in src:
    assert old2 in src and old3 in src, "Không tìm thấy đoạn patch 2"
    src = src.replace(old2, new2).replace(old3, new3)
open(path, "w").write(src)
print("Patch OK:", "M3DIFF_QUANT" in src, "[patch] resume" in src)

## Bước 5 — Ghi các script lên Volume
Dùng lại code của Notebook 1 ở `/mnt/m3diff/synth/code` (dán vật, đổi màu, chấm câu).

**ft_common.py** — Bộ nhận diện câu nói về camera + kiểm định McNemar

In [ ]:
%%writefile /mnt/m3diff/ft/code/ft_common.py
"""Hàm chung cho phần đánh giá fine-tune."""
import re

from scipy.stats import binomtest

CAM = re.compile(
    r"\b(perspective|viewpoint|point of view|viewing angle|shooting angle|camera angle|field of view|"
    r"zoom(ed|s|ing)?|panned|"
    r"camera (has |had )?(been )?(moved|shifted|panned|rotated|tilted|turned|zoomed)|"
    r"(the )?view (has )?(shifted|moved|panned|rotated|tilted))\b", re.I)   # không dùng 'tilted'/'frame' đứng riêng: hay là tư thế vật / khung tranh
ZOOM_IN = re.compile(r"\bzoom(ed|s|ing)? in\b|\bcloser\b", re.I)
ZOOM_OUT = re.compile(r"\bzoom(ed|s|ing)? out\b|\bfarther\b|\bmore of the scene\b", re.I)
CW = re.compile(r"\bclockwise\b", re.I)
CCW = re.compile(r"\b(counter-?clockwise|anti-?clockwise)\b", re.I)


def camera_sents(caption):
    return [s for s in re.split(r"(?<=[.!?;])\s+", caption) if CAM.search(s)]


def camera_dir_ok(caption, cond, direction):
    ss = " ".join(camera_sents(caption))
    if not ss:
        return False
    if cond == "zoom_in" or (direction == "in"):
        return bool(ZOOM_IN.search(ss)) and not ZOOM_OUT.search(ss)
    if cond == "zoom_out" or (direction == "out"):
        return bool(ZOOM_OUT.search(ss)) and not ZOOM_IN.search(ss)
    if direction in ("clockwise", "counterclockwise"):
        ccw = bool(CCW.search(ss))
        cw = bool(CW.search(ss)) and not ccw
        return ccw if direction == "counterclockwise" else cw
    if direction in ("left", "right"):
        other = "right" if direction == "left" else "left"
        return bool(re.search(rf"\b{direction}\b", ss, re.I)) and not re.search(rf"\b{other}\b", ss, re.I)
    return False


def mcnemar(a, b):
    """a, b: list bool ghép cặp (model gốc, model fine-tune). p-value McNemar chính xác."""
    n01 = sum(1 for x, y in zip(a, b) if not x and y)
    n10 = sum(1 for x, y in zip(a, b) if x and not y)
    if n01 + n10 == 0:
        return 1.0, n01, n10
    return binomtest(n01, n01 + n10, 0.5).pvalue, n01, n10

**make_ft_data.py** — Sinh dữ liệu train tổng hợp và bộ test camera

In [ ]:
%%writefile /mnt/m3diff/ft/code/make_ft_data.py
"""Sinh dữ liệu tổng hợp cho fine-tune M3Diff và bộ test chuyển động camera.

--mode train   : ảnh nền lấy từ OmniDiff TRAIN, vật thể COCO không trùng bộ test -> train.json (định dạng LLaVA,
                 có câu trả lời kiểu OmniDiff: phần chỉ vật + phần thay đổi, kèm màu và vị trí của vật).
--mode camtest : ảnh nền lấy từ OmniDiff TEST, vật thể của bộ test Notebook 1 -> questions.json + manifest.jsonl
                 (camera zoom vào/ra, xoay, lia; camera + thêm/xoá vật). Dùng để so model gốc và model fine-tune.

Dùng lại các hàm dán vật / đổi màu / gamma của make_synth.py (Notebook 1).
"""
import argparse, glob, json, math, os, sys

import cv2
import numpy as np
import pandas as pd

p0 = argparse.ArgumentParser(add_help=False)
p0.add_argument("--synth-code", default="/mnt/m3diff/synth/code")
_a0, _ = p0.parse_known_args()
sys.path.insert(0, _a0.synth_code)
from make_synth import prepare_object, paste, sample_center, hue_shift, gamma, MARGIN  # noqa: E402

OUT_LONG = 768
SRC_LONG = 1024          # ảnh nguồn to hơn ảnh ra để có chỗ cho zoom/lia/xoay

TRAIN_PLAN = {"add": 380, "remove": 380, "color": 300, "move": 360, "size": 360, "light": 150,
              "zoom": 260, "rotate": 160, "pan": 160, "cam_obj": 160, "same": 50}
CAMTEST_LEVELS = {"zoom_in": [1.12, 1.25, 1.5], "zoom_out": [1.12, 1.25, 1.5],
                  "rotate": [3, 7, 15], "pan": [0.06, 0.12, 0.22]}
CAMTEST_OBJ = {"cam_add": 30, "cam_remove": 30}


def parse():
    p = argparse.ArgumentParser(parents=[p0])
    p.add_argument("--mode", choices=["train", "camtest"], required=True)
    p.add_argument("--omnidiff-root", required=True)
    p.add_argument("--bg-globs", nargs="*", default=[], help="(train) glob ảnh nền, tương đối với omnidiff-root")
    p.add_argument("--questions", nargs="*", default=[], help="(camtest) file câu hỏi OmniDiff test để lấy ảnh nền")
    p.add_argument("--prompt-questions", nargs="+", required=True, help="file câu hỏi OmniDiff để lấy mẫu prompt")
    p.add_argument("--objects-dir", required=True)
    p.add_argument("--objects-csv", default="objects.csv")
    p.add_argument("--out", required=True)
    p.add_argument("--n-bg", type=int, default=300, help="số ảnh nền mỗi nguồn")
    p.add_argument("--scale", type=float, default=1.0, help="(train) nhân số cặp mỗi loại")
    p.add_argument("--n-per-level", type=int, default=15, help="(camtest)")
    p.add_argument("--seed", type=int, default=7001)
    p.add_argument("--rel-prefix", default="", help="tiền tố đường dẫn ảnh ghi vào json (train)")
    return p.parse_args()


# ---------------- mô tả bằng chữ ----------------
def color_name(rgb, a):
    m = a > 0.5
    if m.sum() < 10:
        return ""
    hsv = cv2.cvtColor(np.clip(rgb, 0, 255).astype(np.uint8), cv2.COLOR_BGR2HSV)[m].astype(np.float32)
    h, s, v = hsv[:, 0] * 2.0, hsv[:, 1], hsv[:, 2]
    S, V = float(np.median(s)), float(np.median(v))
    if V < 55:
        return "black"
    if S < 45:
        return "white" if V > 190 else "gray"
    w = s * v + 1e-6
    ang = math.degrees(math.atan2((w * np.sin(np.radians(h))).sum(), (w * np.cos(np.radians(h))).sum())) % 360
    if ang < 15 or ang >= 345:
        return "red"
    if ang < 40:
        return "brown" if V < 150 else "orange"
    if ang < 70:
        return "yellow"
    if ang < 160:
        return "green"
    if ang < 195:
        return "cyan"
    if ang < 255:
        return "blue"
    if ang < 290:
        return "purple"
    return "pink"


def loc_phrase(cx, cy, W, H):
    hx = "left" if cx < W / 3 else ("right" if cx > 2 * W / 3 else "center")
    vy = "upper" if cy < H / 3 else ("lower" if cy > 2 * H / 3 else "middle")
    if hx == "center" and vy == "middle":
        return "in the center of the scene"
    if hx == "center":
        return f"in the {vy} part of the scene"
    if vy == "middle":
        return f"on the {hx} side of the scene"
    return f"in the {vy} {hx} part of the scene"


def np_(c, n):
    return f"{c} {n}" if c else n


def art(word):
    return "An" if word[:1].lower() in "aeiou" else "A"


def cap_add(rng, c, n, loc):
    d = np_(c, n)
    return rng.choice([f"{art(d)} {d} has appeared {loc}.", f"{art(d)} {d} has been added {loc}.",
                       f"{art(d)} new {d} is now present {loc}."])


def cap_remove(rng, c, n, loc):
    d = np_(c, n)
    return rng.choice([f"The {d} {loc} has disappeared.", f"The {d} {loc} is no longer present.",
                       f"The {d} that was {loc} has been removed."])


def cap_color(rng, c1, c2, n, loc):
    if c1 == c2 or not c1 or not c2:
        return f"The {n} {loc} has changed to a different shade of {c1 or 'color'}."
    return rng.choice([f"The {n} {loc} has changed from {c1} to {c2}.",
                       f"The color of the {n} {loc} has changed from {c1} to {c2}.",
                       f"The {c1} {n} {loc} has turned {c2}."])


def cap_move(rng, c, n, loc1, direction):
    d = np_(c, n)
    return rng.choice([f"The {d} {loc1} has moved to the {direction}.",
                       f"The {d} that was {loc1} has been moved to the {direction}.",
                       f"The {d} {loc1} has shifted toward the {direction} side of the scene."])


def cap_size(rng, c, n, loc, up):
    d = np_(c, n)
    if up:
        return rng.choice([f"The {d} {loc} has become larger.", f"The size of the {d} {loc} has increased.",
                           f"The {d} {loc} has been enlarged."])
    return rng.choice([f"The {d} {loc} has become smaller.", f"The size of the {d} {loc} has decreased.",
                       f"The {d} {loc} has shrunk."])


def cap_light(rng, up):
    if up:
        return rng.choice(["The scene has become brighter.", "The ambient lighting has brightened.",
                           "The lighting in the scene has increased."])
    return rng.choice(["The scene has become darker.", "The ambient lighting has dimmed.",
                       "The lighting in the scene has decreased."])


def cap_camera(rng, kind, val, extra):
    small = {"zoom": val < 1.2, "rotate": val < 5, "pan": val < 0.1}.get(kind, False)
    adv = "slightly " if small else ""
    if kind == "zoom":
        io = extra
        return rng.choice([f"The shooting perspective has {adv}zoomed {io}.",
                           f"The camera has {adv}zoomed {io}" + (", making objects appear larger." if io == "in"
                                                                 else ", so more of the scene is visible.")])
    if kind == "rotate":
        return rng.choice([f"The camera has rotated {adv}{extra}.", f"The shooting angle has tilted {adv}{extra}."])
    if kind == "pan":
        return rng.choice([f"The shooting perspective has {adv}shifted to the {extra}.",
                           f"The camera has {adv}panned to the {extra}."])
    raise ValueError(kind)


# ---------------- biến đổi camera ----------------
def fit_out(img):
    return resize_to_long(img, OUT_LONG)


def resize_to_long(img, side):
    s = side / max(img.shape[:2])
    return cv2.resize(img, (max(2, round(img.shape[1] * s)), max(2, round(img.shape[0] * s))),
                      interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)


def cam_zoom(src, f, rng):
    """A = toàn khung, B = cắt giữa 1/f (zoom vào). Tâm lệch ngẫu nhiên nhẹ."""
    H, W = src.shape[:2]
    w, h = W / f, H / f
    cx = W / 2 + rng.uniform(-0.25, 0.25) * (W - w)
    cy = H / 2 + rng.uniform(-0.25, 0.25) * (H - h)
    x0, y0 = int(round(cx - w / 2)), int(round(cy - h / 2))
    B = cv2.resize(src[y0:y0 + int(h), x0:x0 + int(w)], (W, H), interpolation=cv2.INTER_CUBIC)
    return src, B


def cam_rotate(src, deg):
    """Xoay quanh tâm và phóng vừa đủ để không lộ viền đen."""
    H, W = src.shape[:2]
    t = math.radians(abs(deg))
    k = math.cos(t) + math.sin(t) * max(W / H, H / W)
    M = cv2.getRotationMatrix2D((W / 2, H / 2), deg, k)
    return src, cv2.warpAffine(src, M, (W, H), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)


def cam_pan(src, p, right):
    """Hai cửa sổ cùng cỡ lệch nhau p*W theo chiều ngang. right=True: máy lia sang phải (nội dung trôi sang trái)."""
    H, W = src.shape[:2]
    d = int(round(p * W))
    A, B = src[:, :W - d], src[:, d:]
    return (A, B) if right else (B, A)


def camera_pair(src, kind, val, rng):
    """Trả về (A, B, mô tả hướng) — chưa resize."""
    if kind == "zoom":
        io = "in" if val >= 1 else "out"
        f = val if val >= 1 else 1 / val
        A, B = cam_zoom(src, f, rng)
        return (A, B, "in") if io == "in" else (B, A, "out")
    if kind == "rotate":
        A, B = cam_rotate(src, val)
        # góc dương = nội dung quay ngược chiều kim đồng hồ = máy xoay cùng chiều kim đồng hồ
        return A, B, ("clockwise" if val > 0 else "counterclockwise")
    if kind == "pan":
        right = val > 0
        A, B = cam_pan(src, abs(val), right)
        return A, B, ("right" if right else "left")
    raise ValueError(kind)


# ---------------- sinh 1 cặp ----------------
def place(rng, bg, pool, area, center_zone=False):
    o = pool.iloc[int(rng.integers(len(pool)))]
    H, W = bg.shape[:2]
    rgb, a = prepare_object(o["_rgba"], area * W * H)
    if center_zone:      # để vật vẫn nằm trong khung sau khi zoom/lia
        c = (float(rng.uniform(0.35, 0.65) * W), float(rng.uniform(0.45, 0.75) * H))
    else:
        c = sample_center(rng, W, H, a.shape[1], a.shape[0])
        if c is None:
            return None
    img, am = paste(bg, rgb, a, *c)
    if img is None:
        return None
    return o, rgb, a, c, img


def make_one(kind, rng, bgs, objs, objs_color, level=None):
    bi = int(rng.integers(len(bgs)))
    name, src = bgs[bi]
    bg = fit_out(src)
    H, W = bg.shape[:2]
    meta = {"bg_source": name, "cond": kind}

    if kind in ("add", "remove"):
        area = level if level is not None else float(np.exp(rng.uniform(np.log(0.008), np.log(0.12))))
        r = place(rng, bg, objs, area)
        if r is None:
            return None
        o, rgb, a, c, img = r
        cn, loc = color_name(rgb, a), loc_phrase(*c, W, H)
        meta |= {"category": o["category"], "obj_id": int(o["obj_id"]), "area": round(area, 4)}
        if kind == "add":
            return bg, img, cap_add(rng, cn, o["category"], loc), meta
        return img, bg, cap_remove(rng, cn, o["category"], loc), meta

    if kind == "color":
        r = place(rng, bg, objs_color, float(rng.uniform(0.02, 0.07)))
        if r is None:
            return None
        o, rgb, a, c, img = r
        deg = float(rng.uniform(25, 150)) * (1 if rng.integers(2) else -1)
        rgb2 = hue_shift(rgb, deg)
        B, _ = paste(bg, rgb2, a, *c)
        cap = cap_color(rng, color_name(rgb, a), color_name(rgb2, a), o["category"], loc_phrase(*c, W, H))
        meta |= {"category": o["category"], "obj_id": int(o["obj_id"]), "hue_shift": round(deg, 1)}
        return img, B, cap, meta

    if kind == "move":
        r = place(rng, bg, objs, float(rng.uniform(0.015, 0.06)))
        if r is None:
            return None
        o, rgb, a, c, img = r
        frac = float(rng.uniform(0.08, 0.5))
        for sgn in ([1, -1] if rng.integers(2) else [-1, 1]):
            B, _ = paste(bg, rgb, a, c[0] + sgn * frac * W, c[1])
            if B is not None:
                cap = cap_move(rng, color_name(rgb, a), o["category"], loc_phrase(*c, W, H),
                               "right" if sgn > 0 else "left")
                meta |= {"category": o["category"], "obj_id": int(o["obj_id"]), "shift": round(sgn * frac, 3)}
                return img, B, cap, meta
        return None

    if kind == "size":
        base = float(rng.uniform(0.015, 0.05))
        r = place(rng, bg, objs, base)
        if r is None:
            return None
        o, rgb, a, c, img = r
        s = float(rng.choice([rng.uniform(0.5, 0.8), rng.uniform(1.3, 2.2)]))
        rgb2, a2 = prepare_object(o["_rgba"], base * W * H, scale=s)
        B, _ = paste(bg, rgb2, a2, c[0], c[1] + a.shape[0] / 2 - a2.shape[0] / 2)
        if B is None:
            return None
        cap = cap_size(rng, color_name(rgb, a), o["category"], loc_phrase(*c, W, H), s > 1)
        meta |= {"category": o["category"], "obj_id": int(o["obj_id"]), "scale": round(s, 3)}
        return img, B, cap, meta

    if kind == "light":
        g = float(rng.uniform(1.2, 2.4))
        up = bool(rng.integers(2))
        meta |= {"gamma": round(1 / g if up else g, 3)}
        return bg, gamma(bg, 1 / g if up else g), cap_light(rng, up), meta

    if kind == "same":
        B = bg.copy()
        if rng.integers(2):
            ok, enc = cv2.imencode(".jpg", bg, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(70, 95))])
            B = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        return bg, B, "There is no difference between the two images.", meta

    # ---- camera (có thể kèm thêm/xoá vật) ----
    src_big = resize_to_long(src, SRC_LONG)
    obj_change = None
    if kind in ("cam_obj", "cam_add", "cam_remove"):
        obj_change = {"cam_obj": rng.choice(["add", "remove"]), "cam_add": "add", "cam_remove": "remove"}[kind]
        cam_kind = rng.choice(["zoom", "rotate", "pan"])
    else:
        cam_kind = {"zoom_in": "zoom", "zoom_out": "zoom"}.get(kind, kind)

    if level is not None:
        val = level
        if kind == "zoom_out":
            val = 1 / level
        if cam_kind == "rotate":
            val = level * (1 if rng.integers(2) else -1)
        if cam_kind == "pan":
            val = level * (1 if rng.integers(2) else -1)
    else:
        val = {"zoom": (float(rng.uniform(1.08, 1.6)) if rng.integers(2) else 1 / float(rng.uniform(1.08, 1.6))),
               "rotate": float(rng.uniform(2, 18)) * (1 if rng.integers(2) else -1),
               "pan": float(rng.uniform(0.04, 0.25)) * (1 if rng.integers(2) else -1)}[cam_kind]

    with_obj = src_big
    if obj_change:
        r = place(rng, src_big, objs, float(rng.uniform(0.02, 0.06)), center_zone=True)
        if r is None:
            return None
        o, rgb, a, c, with_obj = r
        meta |= {"category": o["category"], "obj_id": int(o["obj_id"]), "obj_change": obj_change}
    win_seed = int(rng.integers(1 << 30))       # cùng cửa sổ camera cho ảnh có vật và không có vật
    A0, B0, extra = camera_pair(src_big, cam_kind, val, np.random.default_rng(win_seed))
    if obj_change:
        Aw, Bw, _ = camera_pair(with_obj, cam_kind, val, np.random.default_rng(win_seed))
        A0, B0 = (A0, Bw) if obj_change == "add" else (Aw, B0)
    A, B = fit_out(A0), fit_out(B0)
    if A.shape != B.shape:
        B = cv2.resize(B, (A.shape[1], A.shape[0]), interpolation=cv2.INTER_AREA)
    cap = cap_camera(rng, cam_kind, abs(val) if cam_kind != "zoom" else (val if val >= 1 else 1 / val), extra)
    if obj_change:
        cn = color_name(rgb, a)
        loc = "in the center of the scene"
        sent_obj = cap_add(rng, cn, o["category"], loc) if obj_change == "add" else cap_remove(rng, cn, o["category"], loc)
        cap = f"{cap} {sent_obj}"
    meta |= {"camera": cam_kind, "camera_value": round(float(val), 4), "camera_dir": extra}
    return A, B, cap, meta


# ---------------- main ----------------
def load_objects(a):
    df = pd.read_csv(os.path.join(a.objects_dir, a.objects_csv))
    df["_rgba"] = [cv2.imread(os.path.join(a.objects_dir, f), cv2.IMREAD_UNCHANGED) for f in df["file"]]
    df = df[df["_rgba"].map(lambda x: x is not None and x.ndim == 3 and x.shape[2] == 4)].reset_index(drop=True)
    return df, df[df["mean_sat"] >= 70].reset_index(drop=True)


def load_bgs(a, rng):
    paths = []
    if a.mode == "train":
        for g in a.bg_globs:
            fs = sorted(glob.glob(os.path.join(a.omnidiff_root, g)))
            paths += [fs[i] for i in rng.permutation(len(fs))[:a.n_bg]]
    else:
        for qf in a.questions:
            uniq = sorted({q["image"][0] for q in json.load(open(qf))})
            paths += [os.path.join(a.omnidiff_root, uniq[i]) for i in rng.permutation(len(uniq))[:a.n_bg]]
    bgs = []
    for pth in paths:
        img = cv2.imread(pth, cv2.IMREAD_COLOR)
        if img is not None:
            bgs.append((os.path.relpath(pth, a.omnidiff_root), resize_to_long(img, SRC_LONG)))
    return bgs


def main():
    a = parse()
    rng0 = np.random.default_rng(a.seed)
    os.makedirs(os.path.join(a.out, "images"), exist_ok=True)
    prompts = sorted({q["conversations"][0]["value"] for qf in a.prompt_questions for q in json.load(open(qf))})
    bgs = load_bgs(a, rng0)
    objs, objs_color = load_objects(a)
    print(f"Ảnh nền: {len(bgs)} | vật thể: {len(objs)} (đủ màu: {len(objs_color)}) | prompt: {len(prompts)}", flush=True)

    plan = []
    if a.mode == "train":
        for k, n in TRAIN_PLAN.items():
            plan += [(k, None, i) for i in range(int(round(n * a.scale)))]
    else:
        for k, levels in CAMTEST_LEVELS.items():
            for li, lv in enumerate(levels):
                plan += [(k, (li, lv), i) for i in range(a.n_per_level)]
        for k, n in CAMTEST_OBJ.items():
            plan += [(k, None, i) for i in range(n)]

    recs, out_json = [], []
    for idx, (kind, lv, i) in enumerate(plan):
        pid = (f"{kind}_L{lv[0]+1}_{i:02d}" if lv else f"{kind}_{i:04d}")
        seed = a.seed * 100000 + idx
        rng = np.random.default_rng(seed)
        res, tries = None, 0
        while res is None and tries < 100:
            res = make_one(kind, rng, bgs, objs, objs_color, level=(lv[1] if lv else None))
            tries += 1
        if res is None:
            print("  bỏ qua", pid); continue
        A, B, cap, meta = res
        pa, pb = f"images/{pid}_A.jpg", f"images/{pid}_B.jpg"
        cv2.imwrite(os.path.join(a.out, pa), A, [cv2.IMWRITE_JPEG_QUALITY, 95])
        cv2.imwrite(os.path.join(a.out, pb), B, [cv2.IMWRITE_JPEG_QUALITY, 95])
        prompt = prompts[int(rng.integers(len(prompts)))]
        rec = {"pair_id": pid, "seed": seed, "image_A": pa, "image_B": pb, "caption": cap, "prompt": prompt,
               "level": (lv[0] + 1) if lv else 0, "level_value": (lv[1] if lv else None)} | meta
        recs.append(rec)
        img = [a.rel_prefix + pa, a.rel_prefix + pb]
        if a.mode == "train":
            out_json.append({"id": f"ftsyn_{pid}", "image": img,
                             "conversations": [{"from": "human", "value": prompt}, {"from": "gpt", "value": cap}]})
        else:
            out_json.append({"id": pid, "image": img, "conversations": [{"from": "human", "value": prompt}]})
        if (idx + 1) % 250 == 0:
            print(f"  {idx+1}/{len(plan)}", flush=True)

    with open(os.path.join(a.out, "manifest.jsonl"), "w") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    name = "train.json" if a.mode == "train" else "questions.json"
    json.dump(out_json, open(os.path.join(a.out, name), "w"), indent=1)
    df = pd.DataFrame(recs)
    print(f"\nĐã sinh {len(df)} cặp -> {a.out}/{name}")
    print(df["cond"].value_counts().to_string())
    print("\nVí dụ câu mô tả:")
    for c in df.groupby("cond").head(1)["caption"]:
        print("  -", c)


if __name__ == "__main__":
    main()

**show_ft_pairs.py** — Lưới kiểm tra cặp ảnh + câu

In [ ]:
%%writefile /mnt/m3diff/ft/code/show_ft_pairs.py
"""Lưới kiểm tra dữ liệu fine-tune: mỗi hàng ảnh A | ảnh B, kèm câu mô tả (hoặc câu model sinh ra)."""
import argparse, json, os, textwrap
os.environ["MPLBACKEND"] = "Agg"
import cv2
import numpy as np
import matplotlib.pyplot as plt

p = argparse.ArgumentParser()
p.add_argument("--root", required=True)
p.add_argument("--out", required=True)
p.add_argument("--n", type=int, default=12)
p.add_argument("--conds", nargs="*", default=None)
p.add_argument("--answers", nargs="*", default=[], help="(tuỳ chọn) nhãn=đường_dẫn.jsonl để in câu model sinh ra")
a = p.parse_args()
recs = [json.loads(l) for l in open(os.path.join(a.root, "manifest.jsonl")) if l.strip()]
if a.conds:
    recs = [r for r in recs if r["cond"] in a.conds]
rng = np.random.default_rng(1)
recs = [recs[i] for i in sorted(rng.permutation(len(recs))[:a.n])]
ans = {}
for spec in a.answers:
    lab, fp = spec.split("=", 1)
    if os.path.exists(fp):
        ans[lab] = {json.loads(l)["id"]: json.loads(l)["caption"] for l in open(fp) if l.strip()}
fig, axes = plt.subplots(len(recs), 2, figsize=(13, 3.9 * len(recs)), facecolor="#fcfcfb", squeeze=False)
for (ax1, ax2), r in zip(axes, recs):
    for ax, k, t in [(ax1, "image_A", "A"), (ax2, "image_B", "B")]:
        ax.imshow(cv2.cvtColor(cv2.imread(os.path.join(a.root, r[k])), cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax1.set_title(f"{r['pair_id']}", loc="left", fontsize=9)
    lines = [f"Đáp án: {r['caption']}"] + [f"{lab}: {d.get(r['pair_id'], '')}" for lab, d in ans.items()]
    ax2.set_title("\n".join(textwrap.fill(s[:300], 95) for s in lines), loc="left", fontsize=8)
fig.tight_layout()
fig.savefig(a.out, dpi=70, facecolor="#fcfcfb")
print(a.out)

**build_mix.py** — Trộn dữ liệu tổng hợp với câu OmniDiff gốc

In [ ]:
%%writefile /mnt/m3diff/ft/code/build_mix.py
"""Ghép dữ liệu fine-tune: cặp tổng hợp + một phần dữ liệu OmniDiff train gốc (để model không quên cách viết câu dài)."""
import argparse, json, os, random

p = argparse.ArgumentParser()
p.add_argument("--synth", required=True, help="train.json của dữ liệu tổng hợp")
p.add_argument("--orig", required=True, help="training_m3diff.json gốc")
p.add_argument("--image-folder", required=True, help="gốc chung của mọi đường dẫn ảnh")
p.add_argument("--n-replay", type=int, default=2700)
p.add_argument("--out", required=True)
p.add_argument("--seed", type=int, default=0)
a = p.parse_args()
rnd = random.Random(a.seed)

syn = json.load(open(a.synth))
orig = json.load(open(a.orig))
omni = [r for r in orig if isinstance(r.get("image"), list) and r["image"][0].startswith("OmniDiff/")]
del orig
ok = [r for r in omni if all(os.path.exists(os.path.join(a.image_folder, i)) for i in r["image"])]
print(f"OmniDiff trong file gốc: {len(omni)} bản ghi | đủ ảnh: {len(ok)}")
real = [r for r in ok if "/Real/" in r["image"][0]]
rend = [r for r in ok if "/Render/" in r["image"][0]]
n_real = min(len(real), round(a.n_replay * 0.55))
n_rend = min(len(rend), a.n_replay - n_real)
replay = rnd.sample(real, n_real) + rnd.sample(rend, n_rend)
for r in replay:
    for c in r["conversations"]:
        if c["from"] == "human" and "<image>" not in c["value"]:
            c["value"] = c["value"].rstrip() + "  <image><image>Difference:"
missing_syn = [r for r in syn if not all(os.path.exists(os.path.join(a.image_folder, i)) for i in r["image"])]
if missing_syn:
    raise SystemExit(f"!! {len(missing_syn)} cặp tổng hợp không thấy ảnh, ví dụ {missing_syn[0]['image']}")
mix = syn + replay
rnd.shuffle(mix)
json.dump(mix, open(a.out, "w"))
print(f"Tổng {len(mix)} mẫu = {len(syn)} tổng hợp + {n_real} OmniDiff-Real + {n_rend} OmniDiff-Render -> {a.out}")
w = [len(c["value"].split()) for r in replay for c in r["conversations"] if c["from"] == "gpt"]
w2 = [len(c["value"].split()) for r in syn for c in r["conversations"] if c["from"] == "gpt"]
print(f"Độ dài câu trả lời: OmniDiff trung vị {sorted(w)[len(w)//2]} từ | tổng hợp trung vị {sorted(w2)[len(w2)//2]} từ")

**ft_train.py** — Chạy train 1 GPU (SDPA, không DeepSpeed), chạy tiếp được

In [ ]:
%%writefile /mnt/m3diff/ft/code/ft_train.py
"""Chạy llava/train/train.py của repo trên 1 GPU, không cần DeepSpeed/FlashAttention.

- Attention dùng SDPA của PyTorch (biến môi trường ATTN, mặc định 'sdpa').
- Không cần DeepSpeed: hàm maybe_zero_3 được thay bằng bản copy thường.
- Sửa chỗ thiếu khi chạy tiếp từ checkpoint: Trainer chỉ nạp lại LoRA, ở đây nạp thêm
  non_lora_trainables.bin (MDP + projector) để không bị lệch với trạng thái optimizer.
"""
import os
import torch
from llava.train import llava_trainer, train as T

_orig = llava_trainer.LLaVATrainer._load_from_checkpoint


def _load(self, resume_from_checkpoint, model=None):
    _orig(self, resume_from_checkpoint, model)
    p = os.path.join(resume_from_checkpoint, "non_lora_trainables.bin")
    if os.path.exists(p):
        sd = torch.load(p, map_location="cpu")
        m = model if model is not None else self.model
        res = m.load_state_dict(sd, strict=False)
        print(f"[resume] nạp lại {len(sd)} tensor MDP/projector, thiếu khớp: {len(res.unexpected_keys)}", flush=True)


llava_trainer.LLaVATrainer._load_from_checkpoint = _load
# Chạy 1 GPU không DeepSpeed: thay hàm gom tham số ZeRO-3 bằng bản copy thường
T.maybe_zero_3 = lambda param, ignore_status=False, name=None: param.detach().cpu().clone()

if __name__ == "__main__":
    T.train(attn_implementation=os.environ.get("ATTN", "sdpa"))

**score_camera.py** — So gốc vs fine-tune trên bộ test camera

In [ ]:
%%writefile /mnt/m3diff/ft/code/score_camera.py
"""So model gốc và model fine-tune trên bộ test chuyển động camera."""
import argparse, json, os, sys

import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from ft_common import CAM, camera_dir_ok, mcnemar  # noqa: E402

p = argparse.ArgumentParser()
p.add_argument("--root", required=True)
p.add_argument("--base", required=True)
p.add_argument("--ft", required=True)
p.add_argument("--synth-code", default="/mnt/m3diff/synth/code")
p.add_argument("--out", required=True)
a = p.parse_args()
sys.path.insert(0, a.synth_code)
from score_captions import classify, mentions_object  # noqa: E402

man = pd.DataFrame([json.loads(l) for l in open(os.path.join(a.root, "manifest.jsonl")) if l.strip()])
def load(fp):
    return {json.loads(l)["id"]: json.loads(l)["caption"] for l in open(fp) if l.strip()}
caps = {"base": load(a.base), "ft": load(a.ft)}
rows = []
for _, r in man.iterrows():
    for m, c in caps.items():
        cap = c.get(r["pair_id"])
        if cap is None:
            continue
        lab = classify(cap)
        d = {"pair_id": r["pair_id"], "cond": r["cond"], "level": r["level"], "model": m, "caption": cap,
             "cam_ok": bool(CAM.search(cap)), "dir_ok": camera_dir_ok(cap, r["cond"], r.get("camera_dir")),
             "false_obj": any(lab[k] for k in ["add", "remove", "color", "move", "size"])}
        if r["cond"] in ("cam_add", "cam_remove"):
            t = "add" if r["cond"] == "cam_add" else "remove"
            d["obj_type_ok"] = lab[t]
            d["obj_name_ok"] = mentions_object(cap, r.get("category"))
            d["false_obj"] = None
        rows.append(d)
df = pd.DataFrame(rows)
df.to_csv(a.out, index=False)

def pct(s):
    s = s.dropna()
    return f"{s.mean()*100:5.1f}" if len(s) else "   —"

print("== Chỉ camera chuyển động (không có vật nào đổi) ==")
print(f"{'Điều kiện':<11}{'Mức':>4}{'n':>4} | {'nhắc camera':>12} {'đúng hướng':>11} {'bịa đổi vật':>12} | (gốc → fine-tune)")
for (cond, lvl), g in df[~df.cond.isin(["cam_add", "cam_remove"])].groupby(["cond", "level"]):
    b, f = g[g.model == "base"], g[g.model == "ft"]
    print(f"{cond:<11}{lvl:>4}{len(b):>4} | {pct(b.cam_ok)} → {pct(f.cam_ok)}  {pct(b.dir_ok)} → {pct(f.dir_ok)}"
          f"  {pct(b.false_obj)} → {pct(f.false_obj)}")
cam_only = df[~df.cond.isin(["cam_add", "cam_remove"])]
b = cam_only[cam_only.model == "base"].set_index("pair_id")
f = cam_only[cam_only.model == "ft"].set_index("pair_id")
ids = b.index.intersection(f.index)
for col, vi in [("cam_ok", "nhắc camera"), ("dir_ok", "đúng hướng"), ("false_obj", "bịa đổi vật")]:
    pv, n01, n10 = mcnemar(b.loc[ids, col].astype(bool).tolist(), f.loc[ids, col].astype(bool).tolist())
    print(f"Tổng {len(ids)} cặp — {vi}: {b.loc[ids, col].mean()*100:.1f}% → {f.loc[ids, col].mean()*100:.1f}%"
          f"  (chỉ fine-tune đúng {n01}, chỉ gốc đúng {n10}, McNemar p = {pv:.2g})")

print("\n== Camera chuyển động + thêm/xoá vật ==")
for cond, g in df[df.cond.isin(["cam_add", "cam_remove"])].groupby("cond"):
    b, f = g[g.model == "base"], g[g.model == "ft"]
    print(f"{cond:<11} n={len(b):<3} đúng loại thay đổi vật {pct(b.obj_type_ok)} → {pct(f.obj_type_ok)} | "
          f"đúng tên vật {pct(b.obj_name_ok)} → {pct(f.obj_name_ok)} | nhắc camera {pct(b.cam_ok)} → {pct(f.cam_ok)}")
print(f"\nĐã lưu: {a.out}")

**compare_synth.py** — So gốc vs fine-tune trên bộ tổng hợp Notebook 1

In [ ]:
%%writefile /mnt/m3diff/ft/code/compare_synth.py
"""So model gốc và fine-tune trên bộ test tổng hợp của Notebook 1 (cùng 600 cặp, cùng bộ chấm)."""
import argparse, os, sys

import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from ft_common import mcnemar  # noqa: E402

p = argparse.ArgumentParser()
p.add_argument("--base", required=True, help="per_pair.csv của model gốc (Notebook 1)")
p.add_argument("--ft", required=True, help="per_pair.csv của model fine-tune")
a = p.parse_args()
b = pd.read_csv(a.base).set_index("pair_id")
f = pd.read_csv(a.ft).set_index("pair_id")
ids = b.index.intersection(f.index)
b, f = b.loc[ids], f.loc[ids]
VI = {"add": "Thêm vật", "remove": "Xoá vật", "color": "Đổi màu", "move": "Dời vật", "size": "Đổi kích thước",
      "light": "Đổi sáng", "same": "Đối chứng A == B", "jpeg": "Đối chứng JPEG"}
print(f"{'Loại':<18}{'n':>4} | {'đúng loại':^22} | {'đúng cả loại + vật':^22}")
for t in ["add", "remove", "color", "move", "size", "light"]:
    m = b["cond"] == t
    if not m.any():
        continue
    x, y = b.loc[m, "type_ok"].astype(bool), f.loc[m, "type_ok"].astype(bool)
    pv, *_ = mcnemar(x.tolist(), y.tolist())
    s = f"{VI[t]:<18}{m.sum():>4} | {x.mean()*100:5.1f} → {y.mean()*100:5.1f} (p={pv:.2g})"
    if t != "light":
        x2, y2 = b.loc[m, "joint_ok"].astype(bool), f.loc[m, "joint_ok"].astype(bool)
        pv2, *_ = mcnemar(x2.tolist(), y2.tolist())
        s += f" | {x2.mean()*100:5.1f} → {y2.mean()*100:5.1f} (p={pv2:.2g})"
    print(s)
print("\nĐối chứng — tỉ lệ bịa thay đổi (càng thấp càng tốt):")
for t in ["same", "jpeg"]:
    m = b["cond"] == t
    if m.any():
        x, y = b.loc[m, "false_alarm"].astype(bool), f.loc[m, "false_alarm"].astype(bool)
        pv, *_ = mcnemar(x.tolist(), y.tolist())
        print(f"  {VI[t]:<18} n={m.sum()} {x.mean()*100:5.1f}% → {y.mean()*100:5.1f}% (p={pv:.2g})")
print("\nTheo mức (đúng loại, gốc → fine-tune):")
for (t, l), g in b[b["cond"].isin(["move", "size", "light", "color"])].groupby(["cond", "level"]):
    print(f"  {VI[t]:<16} mức {l}: {g['type_ok'].mean()*100:5.1f} → {f.loc[g.index, 'type_ok'].mean()*100:5.1f}")

**eval_omnidiff_ft.py** — So gốc vs fine-tune trên OmniDiff (BLEU/METEOR/ROUGE/CIDEr)

In [ ]:
%%writefile /mnt/m3diff/ft/code/eval_omnidiff_ft.py
"""So điểm OmniDiff (BLEU-4, METEOR, ROUGE-L, CIDEr) giữa model gốc và model fine-tune + kiểm tra các cặp zoom thật."""
import argparse, json, os, re, sys
from pathlib import Path

import numpy as np
from scipy.stats import wilcoxon

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from ft_common import CAM  # noqa: E402

p = argparse.ArgumentParser()
p.add_argument("--repo", required=True)
p.add_argument("--split", required=True)
p.add_argument("--base", required=True)
p.add_argument("--ft", required=True)
a = p.parse_args()
sys.path.insert(0, a.repo)
from metrics import load_predictions  # noqa: E402
from pycocotools.coco import COCO  # noqa: E402
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer  # noqa: E402
from pycocoevalcap.bleu.bleu import Bleu  # noqa: E402
from pycocoevalcap.meteor.meteor import Meteor  # noqa: E402
from pycocoevalcap.rouge.rouge import Rouge  # noqa: E402
from pycocoevalcap.cider.cider import Cider  # noqa: E402

ref = os.path.join(a.repo, f"data/evaluation/reference/OmniDiff_{a.split}.json")
coco = COCO(ref)


def score(pred):
    res = coco.loadRes(load_predictions(Path(pred)))
    ids = res.getImgIds()
    tok = PTBTokenizer()
    gts = tok.tokenize({i: coco.imgToAnns[i] for i in ids})
    rs = tok.tokenize({i: res.imgToAnns[i] for i in ids})
    out, per = {}, {}
    for sc, names in [(Bleu(4), ["B1", "B2", "B3", "Bleu_4"]), (Meteor(), ["METEOR"]), (Rouge(), ["ROUGE_L"]), (Cider(), ["CIDEr"])]:
        s, ps = sc.compute_score(gts, rs)
        s = s if isinstance(s, list) else [s]
        for n, v in zip(names, s):
            out[n] = float(v)
        if names[-1] == "CIDEr":
            per = dict(zip(list(gts.keys()), np.asarray(ps, float)))
    return out, per


cb, pb = score(a.base)
cf, pf = score(a.ft)
print(f"\n== OmniDiff-{a.split} ==")
print(f"{'Chỉ số':<9}{'Gốc':>8}{'Fine-tune':>11}{'Chênh':>8}")
for k in ["Bleu_4", "METEOR", "ROUGE_L", "CIDEr"]:
    print(f"{k:<9}{cb[k]*100:>8.1f}{cf[k]*100:>11.1f}{(cf[k]-cb[k])*100:>+8.1f}")
ids = sorted(set(pb) & set(pf))
x, y = np.array([pb[i] for i in ids]), np.array([pf[i] for i in ids])
print(f"CIDEr từng câu: tăng {int((y > x).sum())}, giảm {int((y < x).sum())}, Wilcoxon p = {wilcoxon(y, x).pvalue:.3g}")

load = lambda fp: {json.loads(l)["image_id"]: json.loads(l)["caption"] for l in open(fp) if l.strip()}
B, F = load(a.base), load(a.ft)
lens = lambda d: np.median([len(v.split()) for v in d.values()])
print(f"Độ dài câu trung vị: gốc {lens(B):.0f} từ, fine-tune {lens(F):.0f} từ")
refs = {r["image_id"]: r["caption"] for r in json.load(open(ref))["annotations"]}
zoom = [k for k, v in refs.items() if re.search(r"\b(perspective|zoom|viewpoint|camera angle|shooting angle)\b", v, re.I)]
print(f"Cặp có câu tham chiếu nói về camera/góc chụp: {len(zoom)} | model nhắc camera: gốc {sum(bool(CAM.search(B.get(k, ''))) for k in zoom)}, "
      f"fine-tune {sum(bool(CAM.search(F.get(k, ''))) for k in zoom)}")
others = [k for k in refs if k not in zoom]
print(f"Cặp KHÔNG có camera trong tham chiếu: model nhắc camera (bịa): gốc {sum(bool(CAM.search(B.get(k, ''))) for k in others)}/{len(others)}, "
      f"fine-tune {sum(bool(CAM.search(F.get(k, ''))) for k in others)}/{len(others)}")

**Hàm dùng chung** (chạy eval, chạy train) — chạy cell này mỗi lần mở lại notebook.

In [ ]:
from IPython.display import Image, display
import pandas as pd
os.makedirs(f"{REPO}/datasets", exist_ok=True)
link = f"{REPO}/datasets/ft_synth"          # ảnh train tổng hợp nằm trên Volume, nối vào thư mục datasets
if os.path.isdir(TRAIN_SYN) and not os.path.islink(link):
    os.symlink(TRAIN_SYN, link)

def run_eval(qfile, answers, image_folder, lora):
    mp = f"--model-path {OUT} --model-base {CKPT}" if lora else f"--model-path {CKPT}"
    cmd = (f"cd {REPO} && PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py "
           f"{mp} --question-file {qfile} --answers-file {answers} --image-folder {image_folder} "
           f"--num-chunks 1 --chunk-idx 0 --temperature 0 --torch-dtype {DTYPE} --no-flash-attention")
    n_total = len(json.load(open(qfile)))
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    if done >= n_total:
        print(f"Đã đủ {done}/{n_total}: {answers}"); return
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True).returncode
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    print(f"-> {done}/{n_total} câu, {(time.time()-t0)/60:.1f} phút, mã thoát {rc}")

MDP_ARGS = '{"num_refinement_layers":2,"feature_dim":1152,"ffn_hidden_dim":4608,"attention_downsample_rate":1}'

def train_args(out_dir, extra=()):
    return [PY, f"{CODE}/ft_train.py",
        "--model_name_or_path", CKPT, "--version", "qwen_2", "--output_dir", out_dir,
        "--data_path", MIX, "--image_folder", f"{REPO}/datasets", "--image_aspect_ratio", "pad",
        "--vision_tower", "google/siglip-so400m-patch14-384", "--mm_vision_select_layer", "[-11,-8,-5,-2]",
        "--unfreeze_mm_vision_tower", "False", "--mm_projector_type", "mlp2x_gelu", "--mm_projector_lr", str(PROJ_LR),
        "--mm_patch_merge_type", "flat", "--mm_resampler_type", "mdp", "--mdp_args", MDP_ARGS,
        "--tune_mm_vision_resampler", "True", "--mm_use_im_start_end", "False", "--mm_use_im_patch_token", "False",
        "--lora_enable", "True", "--lora_r", str(LORA_R), "--lora_alpha", str(LORA_ALPHA),
        "--num_train_epochs", str(EPOCHS), "--per_device_train_batch_size", str(BATCH),
        "--gradient_accumulation_steps", str(ACCUM), "--learning_rate", str(LR), "--weight_decay", "0.",
        "--warmup_ratio", "0.03", "--lr_scheduler_type", "cosine", "--logging_steps", "5",
        "--model_max_length", "4096", "--gradient_checkpointing", "True", "--group_by_modality_length", "True",
        "--dataloader_num_workers", "4", "--lazy_preprocess", "True", "--bf16", "True", "--tf32", "True",
        "--report_to", "none", *extra]

def run(args):
    from collections import deque
    env = dict(os.environ, ATTN="sdpa", PYTHONWARNINGS="ignore", TOKENIZERS_PARALLELISM="false")
    t0, tail = time.time(), deque(maxlen=40)
    p = subprocess.Popen(args, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    keep = ("'loss'", "train_runtime", "out of memory", "resume", "it/s", "s/it", "Adding LoRA")
    for line in p.stdout:
        tail.append(line.rstrip())
        if any(k in line for k in keep):
            print(line.rstrip()[-300:], flush=True)
    p.wait()
    if p.returncode != 0:
        print("\n----- LỖI: 40 dòng cuối -----\n" + "\n".join(tail))
    print(f"[xong] mã thoát {p.returncode}, {(time.time()-t0)/60:.1f} phút")
    return p.returncode
print("OK")

## Bước 6 — Vật thể COCO cho tập train (không trùng vật của bộ test)
Lấy tối đa 45 vật/lớp, rồi **loại các vật đã dùng trong bộ test** của Notebook 1.

In [ ]:
if not os.path.exists(f"{OBJ_ALL}/objects.csv"):
    !{PY} {SYN}/code/prep_coco_objects.py --cache {SYN}/coco_cache --out {OBJ_ALL} --per-cat 45
allo = pd.read_csv(f"{OBJ_ALL}/objects.csv")
test_ids = set(pd.read_csv(f"{SYN}/objects/objects.csv")["obj_id"])
train_o = allo[~allo["obj_id"].isin(test_ids)]
train_o.to_csv(f"{OBJ_ALL}/objects_train.csv", index=False)
print(f"Vật thể: tất cả {len(allo)} | trùng bộ test {len(allo)-len(train_o)} (đã loại) | dùng để train {len(train_o)}")
print(train_o["category"].value_counts().to_string())

## Bước 7 — Sinh dữ liệu train tổng hợp + bộ test camera (~10 phút, không dùng GPU)
- **Train** (nền OmniDiff *train*): thêm, xoá, đổi màu, dời, đổi kích thước, đổi sáng, camera zoom/xoay/lia, camera + thêm/xoá vật, không đổi.
  Câu mô tả viết theo kiểu OmniDiff: *phần chỉ vật (màu + tên + vị trí) + phần thay đổi*, mỗi loại vài cách diễn đạt.
- **Test camera** (nền OmniDiff *test*, vật của bộ test): zoom vào / zoom ra / xoay / lia × 3 mức × 15 cặp + camera kèm thêm/xoá vật 60 cặp = 240 cặp.

In [ ]:
QF = f"{REPO}/data/evaluation/prompt/OmniDiff_Real_test.json {REPO}/data/evaluation/prompt/OmniDiff_Render_test.json"
if not os.path.exists(f"{TRAIN_SYN}/train.json"):
    !{PY} {CODE}/make_ft_data.py --synth-code {SYN}/code --mode train --omnidiff-root {REPO}/datasets/OmniDiff \
      --bg-globs "Real/train/images/*" "Render/train/images/*" --n-bg 300 --prompt-questions {QF} \
      --objects-dir {OBJ_ALL} --objects-csv objects_train.csv --out {TRAIN_SYN} --scale {SCALE} --rel-prefix ft_synth/
if not os.path.exists(f"{CAMTEST}/questions.json"):
    !{PY} {CODE}/make_ft_data.py --synth-code {SYN}/code --mode camtest --omnidiff-root {REPO}/datasets/OmniDiff \
      --questions {QF} --n-bg 60 --prompt-questions {QF} --objects-dir {SYN}/objects --out {CAMTEST} --n-per-level 15

!{PY} {CODE}/show_ft_pairs.py --root {TRAIN_SYN} --out {FT}/results/train_examples.png --n 12
display(Image(f"{FT}/results/train_examples.png"))
!{PY} {CODE}/show_ft_pairs.py --root {CAMTEST} --out {FT}/results/camtest_examples.png --n 8
display(Image(f"{FT}/results/camtest_examples.png"))

Kiểm tra bằng mắt: zoom vào thì ảnh B phải là phần giữa phóng to của A; lia phải thì nội dung trôi sang trái; câu đáp án khớp với ảnh.

## Bước 8 — Trộn dữ liệu train

In [ ]:
if not os.path.islink(f"{REPO}/datasets/ft_synth"):
    os.symlink(TRAIN_SYN, f"{REPO}/datasets/ft_synth")
!{PY} {CODE}/build_mix.py --synth {TRAIN_SYN}/train.json --orig {FT}/orig/training_m3diff.json \
  --image-folder {REPO}/datasets --n-replay {N_REPLAY} --out {MIX}

## Bước 9 — Chạy model GỐC trên bộ test camera (~8 phút)
Để có số đối chiếu. (Bộ tổng hợp 600 cặp và OmniDiff đã có kết quả gốc từ trước.)

In [ ]:
run_eval(f"{CAMTEST}/questions.json", f"{EVAL}/camtest_base.jsonl", CAMTEST, lora=False)

## Bước 10 — Train thử 3 bước (~5 phút)
Kiểm tra nạp model, bộ nhớ GPU và tốc độ trước khi train thật. Xem dòng cuối có `train_runtime` và không có lỗi `out of memory`.

In [ ]:
SMOKE = f"{FT}/smoke-m3diff-lora"
shutil.rmtree(SMOKE, ignore_errors=True)
rc = run(train_args(SMOKE, ["--max_steps", "3", "--save_strategy", "no"]))
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
shutil.rmtree(SMOKE, ignore_errors=True)

Nếu báo **out of memory**: đặt `BATCH = BATCH // 2; ACCUM = ACCUM * 2` ở cell mới rồi chạy lại cell trên.
Thời gian train thật ≈ (số mẫu / 16) bước × thời gian 1 bước (xem `train_runtime` / 3).

## Bước 11 — Train thật (~1–1,5 giờ trên A100, chạy tiếp được)
Checkpoint lưu mỗi 50 bước vào `/mnt/m3diff/ft/m3diff-ft-lora/`. Bị ngắt thì chạy lại Bước 1–5 rồi cell này, nó tự chạy tiếp.

In [ ]:
if not os.path.islink(f"{REPO}/datasets/ft_synth"):
    os.symlink(TRAIN_SYN, f"{REPO}/datasets/ft_synth")
if os.path.exists(f"{OUT}/adapter_model.bin") or os.path.exists(f"{OUT}/adapter_model.safetensors"):
    print("Đã train xong:", OUT)
else:
    rc = run(train_args(OUT, ["--save_strategy", "steps", "--save_steps", "50", "--save_total_limit", "1"]))

# Đảm bảo khi nạp lại, vision tower lấy đúng trọng số đã fine-tune của M3Diff gốc (nạp sớm từ checkpoint)
cfg_p = f"{OUT}/config.json"
if os.path.exists(cfg_p):
    cfg = json.load(open(cfg_p)); cfg["unfreeze_mm_vision_tower"] = True
    json.dump(cfg, open(cfg_p, "w"), indent=2)
    print("Đã lưu model:", sorted(os.listdir(OUT)))

## Bước 12 — Kiểm tra nhanh model fine-tune trên 8 cặp của Notebook 1 (~3 phút)
Đọc câu sinh ra: phải là câu tiếng Anh bình thường, có nghĩa. Nếu ra chữ lặp vô nghĩa thì dừng lại, gửi mình ảnh.

In [ ]:
san = f"{EVAL}/sanity_ft.jsonl"
if os.path.exists(san):
    os.remove(san)
run_eval(f"{SYN}/sanity/questions.json", san, f"{SYN}/sanity", lora=True)
base_san = {json.loads(l)["id"]: json.loads(l)["caption"] for l in open(f"{SYN}/sanity/answers.jsonl")} if os.path.exists(f"{SYN}/sanity/answers.jsonl") else {}
for l in open(san):
    r = json.loads(l)
    print(f"\n[{r['id']}]\n  gốc      : {base_san.get(r['id'], '—')}\n  fine-tune: {r['caption']}")

## Bước 13 — Chạy model FINE-TUNE trên 4 bộ đánh giá (~1,5 giờ, chạy tiếp được)
Đặt `RUN_RENDER = False` để bỏ Render cho nhanh (~25 phút).

In [ ]:
RUN_RENDER = True
run_eval(f"{CAMTEST}/questions.json", f"{EVAL}/camtest_ft.jsonl", CAMTEST, lora=True)
run_eval(f"{SYN}/data/questions.json", f"{EVAL}/synth_ft.jsonl", f"{SYN}/data", lora=True)
run_eval(f"{REPO}/data/evaluation/prompt/OmniDiff_Real_test.json", f"{EVAL}/Real_ft.jsonl", f"{REPO}/datasets/OmniDiff", lora=True)
if RUN_RENDER and os.path.exists(f"{VOL}/results/OmniDiff_Render/OmniDiff_Render.jsonl"):
    run_eval(f"{REPO}/data/evaluation/prompt/OmniDiff_Render_test.json", f"{EVAL}/Render_ft.jsonl", f"{REPO}/datasets/OmniDiff", lora=True)

## Bước 14 — So sánh model gốc và fine-tune

In [ ]:
if shutil.which("java") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless",
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if shutil.which("java") is None:
    java_home = subprocess.run([PY, "-c", "import jdk; print(jdk.install('11', jre=True, path='/root/.jdk'))"],
                               capture_output=True, text=True).stdout.strip()
    os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]
FILTER = 'grep -v "tokeniz\\|index\\|Loading\\|DONE\\|Done\\|testlen\\|ratio\\|loading\\|creating\\|annotations"'

print("\n############ ① BỘ TEST CAMERA ############")
!{PY} {CODE}/score_camera.py --root {CAMTEST} --base {EVAL}/camtest_base.jsonl --ft {EVAL}/camtest_ft.jsonl \
  --synth-code {SYN}/code --out {FT}/results/camtest_compare.csv

print("\n############ ② BỘ TỔNG HỢP NOTEBOOK 1 (600 cặp) ############")
!{PY} {SYN}/code/score_captions.py --root {SYN}/data --answers {EVAL}/synth_ft.jsonl --out {FT}/results/synth_ft --n-manual 0 > /dev/null
!{PY} {CODE}/compare_synth.py --base {SYN}/results/per_pair.csv --ft {FT}/results/synth_ft/per_pair.csv

print("\n############ ③④ OMNIDIFF TEST ############")
%cd {REPO}
!{PY} {CODE}/eval_omnidiff_ft.py --repo {REPO} --split Real --base {VOL}/results/OmniDiff_Real/OmniDiff_Real.jsonl --ft {EVAL}/Real_ft.jsonl 2>&1 | {FILTER}
if os.path.exists(f"{EVAL}/Render_ft.jsonl"):
    !{PY} {CODE}/eval_omnidiff_ft.py --repo {REPO} --split Render --base {VOL}/results/OmniDiff_Render/OmniDiff_Render.jsonl --ft {EVAL}/Render_ft.jsonl 2>&1 | {FILTER}

Cách đọc:
- **① Camera:** "nhắc camera" và "đúng hướng" càng cao càng tốt; "bịa đổi vật" (nói vật thay đổi trong khi chỉ camera động) càng thấp càng tốt. p < 0,05 là chênh lệch thật.
- **② Bộ tổng hợp:** kỳ vọng tăng ở đổi kích thước, dời vật; thêm/xoá không được tụt.
- **③④ OmniDiff:** điều kiện quan trọng nhất — CIDEr **không được tụt rõ** (Wilcoxon p ≥ 0,05 hoặc tăng). Kèm số cặp zoom thật model nhắc được camera.

Lưu ý khi viết báo cáo: model fine-tune học đúng cách diễn đạt của câu tổng hợp nên bộ chấm từ khoá ở ①② dễ bắt hơn → ①② có phần thuận lợi cho fine-tune; ③④ (câu người viết, model chưa từng thấy) là phép thử công bằng nhất.

In [ ]:
!{PY} {CODE}/show_ft_pairs.py --root {CAMTEST} --out {FT}/results/camtest_compare.png --n 8 \
  --answers "Gốc={EVAL}/camtest_base.jsonl" "Fine-tune={EVAL}/camtest_ft.jsonl"
display(Image(f"{FT}/results/camtest_compare.png"))

## Kết quả lưu ở `/mnt/m3diff/ft/`
- `m3diff-ft-lora/`: model fine-tune (LoRA + MDP + projector, ~1GB) — nạp kèm checkpoint gốc
- `eval/`: câu model sinh ra (gốc và fine-tune) trên các bộ đánh giá
- `results/`: bảng so sánh, ảnh ví dụ
- `train_synth/`, `camtest/`, `train_mix.json`: dữ liệu (có `manifest.jsonl` ghi seed và tham số từng cặp)

Xong nhớ bấm **Terminate**.